### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


# Lecture 11: K-Means Step Demo

Choose initial centroids, run one step at a time, and vary number of clusters $k$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

rng = np.random.default_rng(21)


def make_dataset(n=120, noise=0.45):
    c = np.array([[-2,-1.5],[2,-1.0],[0,2.0]])
    xs=[]
    for center in c:
        xs.append(center + rng.normal(0, noise, (n//3,2)))
    return np.vstack(xs)

state = {'X': make_dataset(), 'centroids': None, 'labels': None}


def init_centroids(k=3, seed=0):
    rr = np.random.default_rng(seed)
    idx = rr.choice(len(state['X']), size=k, replace=False)
    state['centroids'] = state['X'][idx].copy()
    state['labels'] = np.zeros(len(state['X']), dtype=int)


def step_once(_=None):
    X = state['X']
    C = state['centroids']
    if C is None:
        return
    d = ((X[:,None,:]-C[None,:,:])**2).sum(axis=2)
    labels = d.argmin(axis=1)
    newC = np.vstack([X[labels==j].mean(axis=0) if np.any(labels==j) else C[j] for j in range(len(C))])
    state['labels'] = labels
    state['centroids'] = newC
    redraw()


def redraw(_=None):
    X = state['X']
    C = state['centroids']
    y = state['labels']
    plt.figure(figsize=(6.4,6))
    if C is not None:
        plt.scatter(X[:,0], X[:,1], c=y, s=18, cmap='tab10', alpha=0.75)
        plt.scatter(C[:,0], C[:,1], c='black', marker='X', s=180, label='centroids')
    else:
        plt.scatter(X[:,0], X[:,1], s=18, alpha=0.75)
    plt.title('K-means interactive steps')
    plt.grid(alpha=0.25)
    plt.legend(loc='best')
    plt.show()


def reset_data(_=None):
    state['X'] = make_dataset(n=n_slider.value, noise=noise_slider.value)
    init_centroids(k=k_slider.value, seed=seed_slider.value)
    redraw()

k_slider = widgets.IntSlider(description='k', min=2, max=8, value=3, continuous_update=False)
seed_slider = widgets.IntSlider(description='init_seed', min=0, max=50, value=0, continuous_update=False)
n_slider = widgets.IntSlider(description='n_points', min=60, max=300, step=30, value=120, continuous_update=False)
noise_slider = widgets.FloatSlider(description='noise', min=0.15, max=1.2, step=0.05, value=0.45, continuous_update=False)
btn_init = widgets.Button(description='Initialize centroids')
btn_step = widgets.Button(description='Run one step')
btn_reset = widgets.Button(description='Regenerate data')
out = widgets.Output()


def on_init(_):
    init_centroids(k=k_slider.value, seed=seed_slider.value)
    with out:
        out.clear_output(wait=True)
        redraw()


def on_step(_):
    with out:
        out.clear_output(wait=True)
        step_once()


def on_reset(_):
    with out:
        out.clear_output(wait=True)
        reset_data()

btn_init.on_click(on_init)
btn_step.on_click(on_step)
btn_reset.on_click(on_reset)

ui = widgets.VBox([
    widgets.HBox([k_slider, seed_slider]),
    widgets.HBox([n_slider, noise_slider]),
    widgets.HBox([btn_init, btn_step, btn_reset]),
])

display(ui, out)
with out:
    reset_data()
